# Analsis of First Move Win Bias

In our current 1v1 agent matchups, there's clearly a bias towards the player who moves first. We can try to compensate for this by rotating seats per-map (run the same number of games over the same maps, but with the seats traded), and I recommend doing this.

But this is merely moving around this bias, not alleviating it. If there is an 80% win rate bias for the player moving first, we'll need a huge number of games to actually measure differences in "skill" between agents. With agents like Expectiminimax, which have tractability problems, we almost certainly will not be able to do this, therefore we should make some effort to first quantify and then alleviate First Player Move Bias.

## Quantify First Move Win Bias

This first experiment tries to quantify the first move win bias by just playing N games of the Greedy agent versus itself.

If there were no bias, you would think that for enough games we'd see the rate at which the first player wins move towards 50%. This experiment tries to quantify that by running an increasing number of experiments to see what first player win rate this gravitates towards.

So we do the following:

* Run an increasing number of games of greedy vs greed. Capture "percentage of first mover wins".
* Re-run each set N times (a small number to maintain tractabiity).
* We then can form a basic mean and standard deviation for the "first player win rate" (FPWR).
* Plot the results. We hopefully see them gravitate towards a known value.

In [ ]:
from nb_setup import risk_ai_game
import matplotlib.pyplot as plt
import random
import statistics

RUN_COUNTS = [20, 40, 80, 100, 200, 400, 600, 800]
NUM_REPEATS = 200

master_rng = random.Random(12345)

mean_rates = []
std_rates = []

for num_games in RUN_COUNTS:
    batch_rates = []

    for _ in range(NUM_REPEATS):
        first_mover_wins = 0

        # draw independent seeds for this batch
        seeds = [master_rng.randrange(1, 10**9) for _ in range(num_games)]

        for seed in seeds:
            winner = risk_ai_game.run_game(
                risk_ai_game.RiskAIGameOptions(
                    agents=[
                        risk_ai_game.GreedyAgent(0, name="Greedy-P0"),
                        risk_ai_game.GreedyAgent(1, name="Greedy-P1"),
                    ],
                    max_turns=500,
                    verbose=False,
                    random_seed=seed,
                )
            )
            if winner == 0:
                first_mover_wins += 1

        batch_rates.append(100.0 * first_mover_wins / num_games)

    mean_rates.append(statistics.mean(batch_rates))
    std_rates.append(statistics.pstdev(batch_rates))
    print(
        f"{num_games} games: mean={mean_rates[-1]:.2f}%, "
        f"std={std_rates[-1]:.2f}%"
    )

plt.figure(figsize=(8, 4))
plt.errorbar(RUN_COUNTS, mean_rates, yerr=std_rates, marker="o", capsize=4)
plt.xlabel("Games per experiment")
plt.ylabel("First player win rate (%)")
plt.title("Greedy vs Greedy: first-mover win rate with variability")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()